In [ ]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, CLIPProcessor, CLIPModel
from PIL import Image
import os
import json
import torch.nn.functional as F
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm
from CustomCLIPModel import CustomCLIPModel  # Import your custom CLIP model

def set_trainable_parameters(model, trainable_substrings=None, matching_fn=None):
    """ Freeze all parameters in the model, and then selectively unfreeze specific parts. """
    for param in model.parameters():
        param.requires_grad = False

    def is_trainable(name):
        if trainable_substrings and any(sub in name for sub in trainable_substrings):
            return True
        if matching_fn and matching_fn(name):
            return True
        return False

    for name, param in model.named_parameters():
        if is_trainable(name):
            param.requires_grad = True
            # print(f"Unfrozen: {name}")


def extract_assistant_content(caption):
    """ Extract the assistant's response from a caption that may contain 'ASSISTANT:'."""
    if "ASSISTANT:" in caption:
        return caption.split("ASSISTANT:")[-1].strip()
    else:
        return caption.strip()


def test_time_adaptation_for_all_images(
    image_directory,
    start_idx=0,
    end_idx=1004,
    initial_user_prompt="Describe this image",
    model_path="llava-hf/llava-1.5-7b-hf",
    clip_model_path="/home/larry5/project/MH_Post_Hoc/exp/RL-MH/llava-15-logit/clipmodel.pth",
    device_llava="cuda:0",
    device_clip="cuda:1",
    lr=2e-3,
    num_steps=5,
    num_beams=5,
    max_new_tokens=512
):
    image_paths = [os.path.join(image_directory, img) for img in os.listdir(image_directory) if img.endswith(('.jpg', '.jpeg', '.png'))]
    image_ids = [int(os.path.splitext(os.path.basename(img))[0].split('_')[-1]) for img in image_paths]
    image_paths_and_ids = sorted(zip(image_ids, image_paths), key=lambda x: x[0])  # Sort by image id

    # Apply the range from start_idx to end_idx
    image_paths_and_ids = image_paths_and_ids[start_idx:end_idx]
    print(f"Processing images from index {start_idx} to {end_idx}... Total: {len(image_paths_and_ids)} images")
    # Dynamically set the output file names
    intermediate_results_json = f"tta_intermediate_results_{start_idx}_{end_idx}.json"
    baseline_results_json = f"tta_baseline_results_{start_idx}_{end_idx}.json"

    image_paths_and_ids = image_paths_and_ids[:250]  # Limit to first 3 images for testing
    # Store intermediate and final baseline results
    intermediate_results_data = []
    baseline_results_data = []

    for idx, (image_id, image_path) in enumerate(tqdm(image_paths_and_ids, desc="Processing Images")):
        # print(f"\nProcessing image {image_path} (ID {image_id}) [{idx+1}/{len(image_paths_and_ids)}]")

        # Clear GPU cache to avoid memory issues
        torch.cuda.empty_cache()

        # Load models fresh from checkpoint
        llava_model = LlavaForConditionalGeneration.from_pretrained(model_path, torch_dtype=torch.float32).to(device_llava, torch.float32)
        llava_processor = AutoProcessor.from_pretrained(model_path)


        # Load Custom CLIP Model
        clip_model = CustomCLIPModel()
        state_dict = torch.load(clip_model_path)
        clip_model.load_state_dict(state_dict)
        clip_model.to(device_clip)
        clip_model.eval()

        llava_model.train()
        set_trainable_parameters(llava_model, matching_fn=lambda n: "language" in n and "norm" in n)
        # set_trainable_parameters(llava_model, matching_fn=lambda n: "projector" in n)

        optimizer = AdamW(filter(lambda p: p.requires_grad, llava_model.parameters()), lr=lr)

        image_results = {
            "image_id": image_id,
            "image_path": image_path,
            "initial_prompt": initial_user_prompt,
            "steps": []
        }

        image = Image.open(image_path).convert("RGB")

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": initial_user_prompt},
                ],
            }
        ]

        for step in range(num_steps):
            # print(f"Step {step+1}/{num_steps} starting...")

            prompt_str = llava_processor.apply_chat_template(conversation, add_generation_prompt=True)
            inputs = llava_processor(
                images=image,
                text=[prompt_str],
                return_tensors="pt",
                padding=True
            ).to(device_llava)

            input_ids = inputs["input_ids"]
            attention_mask = inputs["attention_mask"]
            pixel_values = inputs["pixel_values"]

            with torch.no_grad():
                generated_ids = llava_model.generate(
                    input_ids=input_ids,
                    pixel_values=pixel_values,
                    attention_mask=attention_mask,
                    max_new_tokens=77,
                    num_beams=num_beams,
                    num_return_sequences=num_beams
                )

            candidates = llava_processor.batch_decode(generated_ids, skip_special_tokens=True)
            # print(f"Beam Candidates at Step {step+1}:", candidates)
            rewards = []
            for candidate in candidates:
                clip_inputs = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")(
                    text=[candidate], 
                    images=image, 
                    return_tensors="pt", 
                    truncation=True, 
                    padding="max_length"
                ).to(device_clip)

                image_inputs = clip_inputs['pixel_values']
                text_inputs = {'input_ids': clip_inputs['input_ids'], 'attention_mask': clip_inputs['attention_mask']}

                with torch.no_grad():
                    _, logits_binary_classification, clip_scores = clip_model.infer(image_inputs, text_inputs)

                clip_score = clip_scores.squeeze(0)
                logits_binary_classification = logits_binary_classification.squeeze(0)

                rewards.append((clip_score.item(), logits_binary_classification.item()))

            # Extract clip_scores and logit scores
            clip_scores, logit_scores = zip(*rewards)
            clip_scores = torch.tensor(clip_scores)
            logit_scores = torch.tensor(logit_scores)

            clip_score_mean = clip_scores.mean()
            logit_score_mean = logit_scores.mean()

            normalized_clip_scores = clip_scores - clip_score_mean
            normalized_logits = logit_scores - logit_score_mean

            combined_rewards = normalized_clip_scores + normalized_logits
            rewards = combined_rewards.to(device_llava, torch.float32)

            assistant_contents = [extract_assistant_content(c) for c in candidates]

            step_data = {
                "step": step+1,
                "beam_captions": assistant_contents,
                "clip_scores": clip_scores.tolist(),
                "rewards": rewards.tolist()
            }
            image_results["steps"].append(step_data)

            if step == 0:
                best_idx_step1 = clip_scores.argmax().item()
                best_caption_step1 = assistant_contents[best_idx_step1]
                best_score_step1 = clip_scores[best_idx_step1].item()
                image_results["initial_best_caption"] = best_caption_step1
                image_results["initial_best_score"] = best_score_step1

            # Compute log probs for RL update
            beam_conversations = []
            for c in assistant_contents:
                beam_conv = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image"},
                            {"type": "text", "text": initial_user_prompt},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": c}
                        ],
                    }
                ]
                beam_conversations.append(beam_conv)

            beam_prompts = [llava_processor.apply_chat_template(conv, add_generation_prompt=False)
                            for conv in beam_conversations]
            # print(f"Beam Prompts at Step {step+1}:", beam_prompts)

            beam_inputs = llava_processor(
                images=image,
                text=beam_prompts,
                return_tensors="pt",
                padding=True
            ).to(device_llava)

            beam_input_ids = beam_inputs["input_ids"]  # [num_beams, seq_len]
            beam_attention_mask = beam_inputs["attention_mask"]  # [num_beams, seq_len]
            beam_pixel_values = pixel_values.repeat(num_beams, 1, 1, 1)

            outputs = llava_model(
                input_ids=beam_input_ids,
                pixel_values=beam_pixel_values,
                attention_mask=beam_attention_mask,
                return_dict=True
            )
            logits = outputs.logits

            logits_float = logits.float()
            log_probs = F.log_softmax(logits_float, dim=-1)

            seq_log_probs = torch.gather(log_probs, 2, beam_input_ids.unsqueeze(-1)).squeeze(-1)
            seq_log_probs_sum = seq_log_probs.sum(dim=1)

            loss = -(rewards * seq_log_probs_sum).mean()

            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(llava_model.parameters(), max_norm=1.0)
            optimizer.step()

        final_prompt_str = llava_processor.apply_chat_template(conversation, add_generation_prompt=True)
        final_inputs = llava_processor(images=image, text=[final_prompt_str], return_tensors="pt", padding=True).to(device_llava)

        with torch.no_grad():
            final_generated = llava_model.generate(
                input_ids=final_inputs["input_ids"],
                pixel_values=final_inputs["pixel_values"],
                attention_mask=final_inputs["attention_mask"],
                max_new_tokens=max_new_tokens
            )

        final_caption = extract_assistant_content(llava_processor.batch_decode(final_generated, skip_special_tokens=True)[0])
        image_results["final_caption"] = final_caption
        # print(f"Final Caption: {final_caption}")


        # Store final result in baseline format
        baseline_results_data.append({
            "id": image_id,
            "response": final_caption
        })

        intermediate_results_data.append(image_results)
        # print(f"Image ID: {image_id}, Final Refined Caption: {final_caption}\n")


        with open(intermediate_results_json, "w") as f:
            json.dump(intermediate_results_data, f, indent=4)

        with open(baseline_results_json, "w") as f:
            json.dump(baseline_results_data, f, indent=4)

        # print(f"Intermediate results saved to {intermediate_results_json}")
        # print(f"Baseline results saved to {baseline_results_json}")


# Example usage
image_directory = "../../data/AMBER/image"
intermediate_results_path = "tta_intermediate_results_250.json"
baseline_results_path = "tta_baseline_results_250.json"

test_time_adaptation_for_all_images(
    image_directory=image_directory,
    start_idx=0, 
    end_idx=250,
    model_path="llava-hf/llava-1.5-7b-hf",
    clip_model_path="../../clipmodel.pth",
    device_llava="cuda:1",
    device_clip="cuda:1",
    lr=2e-3,
    num_steps=5,
    num_beams=5,
    max_new_tokens=512
)